# Five-stock buy-and-hold: offline acceptance

This is a **synthetic accounting demonstration**, not historical investment evidence.
Five hypothetical stocks start with **$100 million of equity** from 2024-01-02 through
2025-01-02. The supplied calendar includes every weekday (including actual exchange
holidays) at 21:00 UTC: one calendar year, **not an NYSE calendar**. Annualization is
an explicit 252-period modeling assumption, independent of the observed row count.

Inputs are committed immutable snapshots. No provider, credential, network connection,
ML model, daily rebalance, dividend reinvestment, or terminal sale is required.
Run all cells using the installed checkout and `.[notebook]`; pinned replay dependencies
are in `requirements-acceptance.txt`. All figures can be customized and exported.

In [ ]:
from pathlib import Path
import json
import math
import platform
import importlib.metadata
import socket

import polars as pl
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
import research_toolkit as rt

# Fail if any code attempts external data acquisition during this replay.
def no_network(*args, **kwargs):
    raise RuntimeError("Acceptance replay is offline")
socket.create_connection = no_network
socket.socket.connect = no_network
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
SNAPSHOTS = ROOT / "examples" / "snapshots" / "v1"
ARTIFACTS = ROOT / "artifacts" / "acceptance"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
market = rt.load_snapshot(SNAPSHOTS / "equities")
benchmark_market = rt.load_snapshot(SNAPSHOTS / "benchmark")
print(json.dumps({"snapshot_id": market.snapshot_id, "source": market.metadata,
                  "benchmark_snapshot_id": benchmark_market.snapshot_id,
                  "benchmark_source": benchmark_market.metadata}, indent=2))
display(market.diagnostics, market.splits, market.dividends)
print("Session observations:", market.sessions.height,
      "holding intervals:", market.sessions.height - 1)

## Configuration and benchmark

Weights are fixed before entry. Entry costs are modeled commissions of 5 bps,
half-spread of 2 bps and impact of 3 bps on actual entry notional. Cash earns 2%
nominal annually; debt costs 6%, both capitalized on calendar dates using ACT/365F.
The research maintenance ratio is 25%, with stop-on-breach and dividend cash repaying debt.

The benchmark is a supplied hypothetical **total-return index** with reinvestment
embedded and no trading costs. Portfolio distributions become receivables and then
cash, with no stock reinvestment. This difference is intentional and visible.
Metrics use a separate 3% effective annual risk-free rate and zero MAR.

In [ ]:
ASSETS = market.prices["asset"].unique().sort().to_list()
INITIAL_CAPITAL = 100_000_000.0
ENTRY, END = market.sessions["session"][0], market.sessions["session"][-1]
LEVERAGES = [1.0, 1.5, 2.0]  # Use [1.0] to omit financed comparisons.
weights = rt.equal_weights(ASSETS)
costs = rt.TradeCosts(commission_bps=5., half_spread_bps=2., impact_bps=3.)
financing = rt.Financing(cash_rate=.02, borrowing_rate=.06, day_count="ACT/365F",
    maintenance_equity_ratio=.25, on_breach="stop", cash_sweep="repay_debt")
metric_options = dict(periods_per_year=252, risk_free_annual_effective=.03,
                      minimum_acceptable_return_annual_effective=0.)
benchmark_returns = rt.returns(benchmark_market, method="simple", basis="total_return")
benchmark = benchmark_returns.values.filter(pl.col("period_start").is_not_null()).select(
    "period_start", "session", "simple_return")
benchmark_metadata = {"source": benchmark_market.metadata["source"],
    "snapshot_id": benchmark_market.snapshot_id, "basis": "total return; reinvested; no costs",
    "currency": "USD", "frequency": "1d"}

def simulate(leverage):
    return rt.buy_and_hold(market, weights=weights, initial_capital=INITIAL_CAPITAL,
        entry_session=ENTRY, end_session=END,
        policy=rt.BuyHoldPolicy(execution="entry_close", sizing="post_cost_equity",
            fractional_shares=True, initial_gross_leverage=leverage, terminal_action="mark_only"),
        costs=costs, financing=financing)

runs = {leverage: simulate(leverage) for leverage in LEVERAGES}
# Stopped results never enter the full-year comparison table.
completed = {leverage: run.require_complete() for leverage, run in runs.items() if run.status == "complete"}
stopped = {leverage: run for leverage, run in runs.items() if run.status != "complete"}
reports = {leverage: rt.performance(run, benchmark=benchmark,
    benchmark_metadata=benchmark_metadata, alignment="strict", **metric_options)
    for leverage, run in completed.items()}
result, report = completed[1.0], reports[1.0]
print(json.dumps(result.metadata, indent=2))
display(pl.concat([r.summary.with_columns(pl.lit(leverage).alias("initial_leverage"))
                   for leverage, r in reports.items()]))
display(report.summary, report.benchmark_comparison)

## Inspect the ledger and daily outputs

The first daily interval includes entry costs once, using the original capital as
its return denominator. Cumulative simple-return sums are displayed alongside,
not substituted for, compounded returns. Every run reconciles inside the simulator;
the checks below independently compare exported balances, positions, P&L and attribution.

In [ ]:
daily_columns = ["session", "pnl", "simple_return", "cumulative_simple_return",
                 "compounded_return", "cumulative_pnl", "equity"]
display(result.daily.select(daily_columns))
result.daily.write_csv(ARTIFACTS / "daily.csv")
display(result.trades, result.costs, result.positions.tail(10), result.receivables.tail(10))
checks = []
for leverage, run in completed.items():
    valuation_residual = run.valuations.select((pl.col("equity") - pl.col("market_value")
        - pl.col("cash") - pl.col("dividend_receivable") + pl.col("debt")).abs().max()).item()
    attribution = run.attribution.group_by("session").agg(pl.col("pnl").sum().alias("attributed_pnl"))
    residual = run.daily.join(attribution, on="session", validate="1:1").select(
        (pl.col("pnl") - pl.col("attributed_pnl")).abs().max()).item()
    final_equity = run.daily["equity"][-1]
    assert valuation_residual < .01 and residual < .01
    assert (run.diagnostics["residual"].abs() <= run.diagnostics["tolerance"]).all()
    assert math.isclose(run.daily["pnl"].sum(), final_equity-INITIAL_CAPITAL, abs_tol=.01)
    assert math.isclose(INITIAL_CAPITAL*(1+run.daily["compounded_return"][-1]), final_equity, abs_tol=.01)
    assert run.trades.height == len(ASSETS)
    assert run.costs.filter(pl.col("trade_id").is_not_null())["date"].unique().to_list() == [ENTRY]
    checks.append({"initial_leverage": leverage, "status": run.status, "ending_equity": final_equity,
        "ending_debt": run.daily["debt"][-1], "ending_leverage": run.daily["gross_leverage"][-1],
        "borrowing_cost": run.costs.filter(pl.col("component") == "borrowing_interest")["amount"].sum(),
        "max_balance_residual": valuation_residual, "max_attribution_residual": residual})
display(pl.DataFrame(checks))
# Identity-preserving disk replay, including raw action inputs.
market = rt.load_snapshot(SNAPSHOTS / "equities")
replayed = simulate(1.0)
assert replayed.daily.equals(result.daily)
print("All complete scenario reconciliations passed.")

## Figures from prepared reports

Raw price split jumps are marked. Charts use prepared tables and never rerun a
simulation. The equity chart includes pre-entry and post-entry valuations, so the
cost at entry remains visible. Allocation includes cash, receivables and negative
debt weights; it is not silently normalized to 100% risky exposure.

In [ ]:
fig, ax = rt.plots.prices(market)
fig.savefig(ARTIFACTS / "prices.png", dpi=130)
display(fig)
plt.close(fig)
for name in ["pnl", "returns", "equity", "drawdown", "distribution", "allocation", "attribution"]:
    fig, ax = getattr(rt.plots, name)(report)
    fig.savefig(ARTIFACTS / f"{name}.png", dpi=130)
    display(fig)
    plt.close(fig)

## Asset correlations with corporate actions accounted for

Each asset below is simulated separately at 1× with zero costs and zero cash
interest. Its splits and dividend receivables are accounted for, and paid dividends
remain in cash. These are **cash-held total portfolio returns**, not split-discontinuous
raw price returns and not an ex-date-reinvested total-return index. Only the
structural leading null is excluded from the common sample.

In [ ]:
asset_tables = []
for asset in ASSETS:
    single = rt.buy_and_hold(market, weights={asset: 1.}, initial_capital=1_000.,
        entry_session=ENTRY, end_session=END,
        policy=rt.BuyHoldPolicy(execution="entry_close", sizing="post_cost_equity",
            fractional_shares=True, initial_gross_leverage=1., terminal_action="mark_only"),
        costs=rt.TradeCosts(commission_bps=0., half_spread_bps=0., impact_bps=0.),
        cash_rate=0., cash_day_count="ACT/365F").require_complete()
    first = pl.DataFrame({"session": [ENTRY], "asset": [asset], "period_start": [None],
                          "simple_return": [None]}, schema={"session": pl.Date, "asset": pl.String,
                                                           "period_start": pl.Date, "simple_return": pl.Float64})
    asset_tables.append(pl.concat([first, single.daily.select("session", pl.lit(asset).alias("asset"),
                                                            "period_start", "simple_return")]))
asset_returns = rt.ReturnResult(pl.concat(asset_tables), metadata={
    "method": "simple", "basis": "total return; dividends held in cash; zero costs/interest",
    "frequency": "1d", "unit": "fraction", "snapshot_id": market.snapshot_id,
    "source": market.metadata, "assets": ASSETS,
    "sessions": [d.isoformat() for d in market.sessions["session"]]}, diagnostics=market.diagnostics)
asset_correlation = rt.correlation(asset_returns)
display(asset_correlation.values)
fig, ax = rt.plots.correlation(asset_correlation)
fig.savefig(ARTIFACTS / "correlation.png", dpi=130)
display(fig)
plt.close(fig)

## Financed scenarios and separately labeled stopped runs

Each scenario reruns the complete ledger. The table above reports actual borrowing
costs and ending leverage, which drifts as equity and debt change. A stopped scenario
is excluded from full-year comparisons. If configuration changes cause a stop,
the following block opts into a report for only its actual coverage and displays
the stop reason. It does not silently trim a full-year benchmark.

In [ ]:
for leverage, stopped_run in stopped.items():
    partial = rt.performance(stopped_run, allow_partial=True, **metric_options)
    display(pl.DataFrame({"initial_leverage": [leverage], "status": [stopped_run.status],
        "stop_reason": [stopped_run.stop_reason], "actual_end": [stopped_run.stop_session],
        "requested_end": [END]}))
    display(partial.summary)
    fig, ax = rt.plots.equity(partial)
    display(fig)
    plt.close(fig)
print("Complete scenarios:", list(completed), "Stopped scenarios:", list(stopped))
# Provenance for this execution; Git information is supplied by the replay script
# when available, rather than invoking subprocesses inside library calculations.
environment = {name: importlib.metadata.version(name) for name in
               ("research-toolkit", "polars", "matplotlib", "nbformat", "nbclient", "ipykernel")}
environment["python"] = platform.python_version()
(ARTIFACTS / "environment.json").write_text(json.dumps(environment, indent=2))
print(environment)

## Observations from the inspected default run

The 262 holding intervals end with about **$111.444 million** at 1×: dollar P&L is
about $11.444 million and the compounded return is **11.4444%**. The sum of daily
simple returns is **11.1488%**, which does not reproduce ending equity. The entry
ledger records five purchases and about **$99,900.10** of transaction costs; there
are no later stock trades.

The net equity and benchmark plots move closely together: return correlation is
**0.9809** and beta is **1.0068**. The hypothetical reinvested benchmark compounds to
**12.2521%**; the portfolio difference is **−0.8077 percentage points**, distinct
from the **−0.7196%** relative-wealth result. The portfolio's maximum drawdown is
about **−7.3490%**. The raw-price plot shows SYNTH_A's 2-for-1 split, while its
share quantity doubles and the equity curve has no mechanical halving.

The 1.5× and 2× runs both complete, ending around **$114.09 million** and
**$116.72 million**, with roughly **$3.05 million** and **$6.13 million** in borrowing
charges. The 2× result is therefore not twice the unlevered return, and its ending
leverage has drifted to about **1.883×**. Its maximum drawdown is about **−14.4445%**.
The maximum independently reconstructed balance and attribution residuals across
these scenarios are below $0.000001. Dividends due after the final session remain
in receivables and contribute to ending equity.

The correlation heatmap is strongly positive across these deliberately related
synthetic paths. It is an accounting/plotting check, not evidence of a diversification
opportunity. The oscillating equity and return plots reflect how this fixture was
constructed. Neither the Sharpe ratio nor these comparisons establish an investable
strategy. Real-data research needs an actual exchange calendar, sourced marks and
actions, licensed data, and independently justified costs/rates. The 252-period
scaling assumption does not match the synthetic calendar's 262 realized intervals.

These observations were written after inspecting the default outputs and exported
figures. If you change configuration or inputs, rerun the calculations and rewrite
this Markdown from the new evidence; calculation code does not generate conclusions.